# Jane Street — GRU with Online Learning
**Based on Evgeniia Grigoreva's #8 solution architecture**

| Component | Detail |
|---|---|
| Model | GRU (PyTorch) with stateful hidden states per symbol |
| Feature Engineering | Rolling statistics over last N time steps per symbol |
| Online Learning | Fine-tune on previous day's realized `responder_6` each morning |
| Ensemble | 3 models with different seeds, averaged at inference |
| Inference | `kaggle_evaluation` API, hidden states persist across batches |

## 1. Setup

In [ ]:
# On Colab, run this first:
# !pip install polars lightgbm -q

import os, gc, time, math, warnings
from typing import Optional
from collections import defaultdict

import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
import glob, pathlib

INPUT_DIR = '/kaggle/input/jane-street-real-time-market-data-forecasting'
# Colab (uncomment if running on Colab after downloading data)
# INPUT_DIR = '/content/jane-street'

# ── Constants ───────────────────────────────────────────────────────────────
TARGET        = 'responder_6'
ALL_RESP      = [f'responder_{i}' for i in range(9)]
N_PARTITIONS  = 10
SEED          = 42
N_ROLL        = 1000   # rolling window size per symbol (matches original)
HIDDEN_SIZE   = 256    # original used 700; reduce to 256 for Colab
NUM_LAYERS    = 2
N_MODELS      = 3      # original used 6; reduce to 3 for Colab
ONLINE_LR     = 1e-4   # learning rate for daily online updates
ONLINE_STEPS  = 3      # gradient steps per online update

torch.manual_seed(SEED)
np.random.seed(SEED)

## 2. Feature Engineering

The core insight from the winning solutions: raw features alone are weak. What matters is the **rolling context** — how each feature is behaving over the last N time steps for this specific symbol. This turns a static snapshot into a temporal fingerprint.

For each feature we compute per-symbol rolling: last value, mean, std.
This triples the feature count but gives the GRU meaningful sequential signal.

In [ ]:
FEATURE_COLS: list[str] = []   # filled after loading one partition

def compute_rolling_features(df: pl.DataFrame, feature_cols: list[str]) -> pl.DataFrame:
    """
    For each feature, compute per-symbol rolling mean and std over the last
    N_ROLL rows. This is the key feature engineering step from the top solutions.

    Input:  raw partition DataFrame
    Output: DataFrame with 3x feature columns (raw + roll_mean + roll_std)
            plus meta columns (date_id, time_id, symbol_id, weight, TARGET)
    """
    roll_exprs = []
    for c in feature_cols:
        roll_exprs += [
            pl.col(c)
              .rolling_mean(window_size=N_ROLL, min_periods=1)
              .over('symbol_id')
              .alias(f'{c}_rmean'),
            pl.col(c)
              .rolling_std(window_size=N_ROLL, min_periods=2)
              .over('symbol_id')
              .fill_null(0.0)
              .alias(f'{c}_rstd'),
        ]
    return df.with_columns(roll_exprs)


def get_model_feature_cols(feature_cols: list[str]) -> list[str]:
    """Return the full list of feature columns fed to the GRU."""
    raw   = feature_cols
    means = [f'{c}_rmean' for c in feature_cols]
    stds  = [f'{c}_rstd'  for c in feature_cols]
    return raw + means + stds

## 3. Sequence Dataset

The GRU is trained on **sequences grouped by (date_id, symbol_id)**. Each sequence is one symbol's full intraday history on one day, ordered by time_id. The GRU reads through the sequence step by step, building up its hidden state, and predicts `responder_6` at each step.

In [ ]:
class JaneStreetSequenceDataset(Dataset):
    """
    Each item is one (date, symbol) sequence.
    Shape of X: (seq_len, n_features)
    Shape of y: (seq_len,)
    Shape of w: (seq_len,)
    """
    def __init__(self, df: pl.DataFrame, feature_cols: list[str]):
        self.sequences = []
        df_pd = df.to_pandas()

        for (date_id, symbol_id), grp in df_pd.groupby(
            ['date_id', 'symbol_id'], sort=False
        ):
            grp = grp.sort_values('time_id')
            X = grp[feature_cols].fillna(0).values.astype(np.float32)
            y = grp[TARGET].fillna(0).values.astype(np.float32)
            w = grp['weight'].fillna(0).values.astype(np.float32)
            self.sequences.append((X, y, w))

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        X, y, w = self.sequences[idx]
        return (
            torch.tensor(X),
            torch.tensor(y),
            torch.tensor(w),
        )


def collate_sequences(batch):
    """Pad sequences to the same length within a batch."""
    Xs, ys, ws = zip(*batch)
    max_len     = max(x.shape[0] for x in Xs)
    n_feat      = Xs[0].shape[1]

    X_pad = torch.zeros(len(Xs), max_len, n_feat)
    y_pad = torch.zeros(len(Xs), max_len)
    w_pad = torch.zeros(len(Xs), max_len)
    mask  = torch.zeros(len(Xs), max_len, dtype=torch.bool)

    for i, (x, y, w) in enumerate(zip(Xs, ys, ws)):
        L = x.shape[0]
        X_pad[i, :L] = x
        y_pad[i, :L] = y
        w_pad[i, :L] = w
        mask[i,  :L] = True

    return X_pad, y_pad, w_pad, mask

## 4. GRU Model

Architecture mirrors Evgeniia's design:

```
features (n_feat) 
   → Linear input projection
   → GRU (hidden_size, num_layers, dropout)
   → Linear output head → scalar prediction
   → tanh × 5  (clips output to [-5, 5], matching target range)
```

The `forward()` method accepts an optional `hidden` state — this is what allows stateful inference: the hidden state from the last batch is passed into the next batch, so the model's memory is never reset mid-day.

In [ ]:
class JaneStreetGRU(nn.Module):
    def __init__(
        self,
        n_features:  int,
        hidden_size: int = HIDDEN_SIZE,
        num_layers:  int = NUM_LAYERS,
        dropout:     float = 0.1,
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        # Input projection: map raw features to a denser representation
        # before feeding to GRU (standard practice in top solutions)
        self.input_proj = nn.Sequential(
            nn.Linear(n_features, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU(),
        )

        self.gru = nn.GRU(
            input_size  = hidden_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0,
        )

        self.head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Linear(hidden_size // 2, 1),
        )

    def forward(
        self,
        x:      torch.Tensor,          # (batch, seq_len, n_features)
        hidden: torch.Tensor = None,   # (num_layers, batch, hidden_size)
    ):
        x_proj = self.input_proj(x)            # (batch, seq_len, hidden_size)
        out, hidden = self.gru(x_proj, hidden) # out: (batch, seq_len, hidden_size)
        pred = self.head(out).squeeze(-1)      # (batch, seq_len)
        pred = torch.tanh(pred) * 5.0          # clip to [-5, 5]
        return pred, hidden


def weighted_r2_torch(
    y_true: torch.Tensor,
    y_pred: torch.Tensor,
    weights: torch.Tensor,
    mask: torch.Tensor,
) -> torch.Tensor:
    """Zero-centered weighted R² — the competition metric."""
    y_true = y_true[mask]
    y_pred = y_pred[mask]
    weights = weights[mask]
    ss_res = (weights * (y_true - y_pred) ** 2).sum()
    ss_tot = (weights * y_true ** 2).sum()
    return 1.0 - ss_res / (ss_tot + 1e-9)


def weighted_mse_loss(
    y_true: torch.Tensor,
    y_pred: torch.Tensor,
    weights: torch.Tensor,
    mask: torch.Tensor,
) -> torch.Tensor:
    """Weighted MSE — what we actually optimize during training."""
    y_true   = y_true[mask]
    y_pred   = y_pred[mask]
    weights  = weights[mask]
    weights  = weights / (weights.sum() + 1e-9)
    return (weights * (y_true - y_pred) ** 2).sum()

## 5. Training

Training strategy:
- Load partitions 0–7 as training data, partition 8–9 as validation
- Compute rolling features on each partition
- Build sequence dataset: one sequence per (date, symbol)
- Train with AdamW + cosine LR schedule
- Train N_MODELS times with different seeds to create an ensemble

In [ ]:
def load_partition(pid: int) -> pl.DataFrame:
    exact = f'{INPUT_DIR}/train.parquet/partition_id={pid}/part-0.parquet'
    glob_ = f'{INPUT_DIR}/train.parquet/partition_id={pid}/*.parquet'
    path  = exact if pathlib.Path(exact).exists() else glob_
    keep  = ['date_id', 'time_id', 'symbol_id', 'weight'] + \
            [f'feature_{i:02d}' for i in range(79)] + ALL_RESP
    return (
        pl.scan_parquet(path, hive_partitioning=False)
        .select(keep)
        .with_columns([
            pl.col(c).cast(pl.Float32)
            for c in [f'feature_{i:02d}' for i in range(79)] + ALL_RESP + ['weight']
        ])
        .collect()
    )


# Load and inspect one partition to get feature names
print('Loading partition 0 to inspect schema...')
df0 = load_partition(0)
FEATURE_COLS = sorted([c for c in df0.columns if c.startswith('feature_')])
MODEL_FEAT   = get_model_feature_cols(FEATURE_COLS)
N_FEATURES   = len(MODEL_FEAT)
print(f'Raw features    : {len(FEATURE_COLS)}')
print(f'Model features  : {N_FEATURES}  (raw + rolling mean + rolling std)')
print(f'Partition 0 rows: {len(df0):,}')
del df0; gc.collect()

In [ ]:
def build_dataset_from_partitions(
    pids: list[int],
    feature_cols: list[str],
    model_feat: list[str],
) -> JaneStreetSequenceDataset:
    """Load partitions, compute rolling features, build sequence dataset."""
    dfs = []
    for pid in pids:
        part = load_partition(pid)
        part = compute_rolling_features(part, feature_cols)
        # Fill nulls (rolling stats are null for first rows)
        part = part.with_columns([
            pl.col(c).fill_null(0.0) for c in model_feat
        ])
        dfs.append(part)
        print(f'  [{pid:02d}] {len(part):>8,} rows')
        del part; gc.collect()
    df = pl.concat(dfs)
    del dfs; gc.collect()
    print(f'  Total: {len(df):,} rows  Est RAM: {df.estimated_size("gb"):.2f} GB')
    return JaneStreetSequenceDataset(df, model_feat)

In [ ]:
def train_one_model(
    model:      JaneStreetGRU,
    train_ds:   JaneStreetSequenceDataset,
    val_ds:     JaneStreetSequenceDataset,
    n_epochs:   int = 5,
    batch_size: int = 128,
    lr:         float = 3e-4,
) -> JaneStreetGRU:

    train_dl = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        collate_fn=collate_sequences, num_workers=2, pin_memory=True
    )
    val_dl = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        collate_fn=collate_sequences, num_workers=2
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=n_epochs
    )

    for epoch in range(n_epochs):
        # ── Train ──────────────────────────────────────────────────────
        model.train()
        train_loss = 0.0
        t0 = time.time()

        for X, y, w, mask in train_dl:
            X, y, w, mask = X.to(DEVICE), y.to(DEVICE), w.to(DEVICE), mask.to(DEVICE)
            optimizer.zero_grad()
            pred, _ = model(X)          # hidden=None: reset each sequence
            loss = weighted_mse_loss(y, pred, w, mask)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()

        # ── Validate ────────────────────────────────────────────────────
        model.eval()
        val_r2 = 0.0
        n_batches = 0

        with torch.no_grad():
            for X, y, w, mask in val_dl:
                X, y, w, mask = X.to(DEVICE), y.to(DEVICE), w.to(DEVICE), mask.to(DEVICE)
                pred, _ = model(X)
                val_r2 += weighted_r2_torch(y, pred, w, mask).item()
                n_batches += 1

        scheduler.step()
        print(
            f'  Epoch {epoch+1}/{n_epochs}  '
            f'loss={train_loss/len(train_dl):.5f}  '
            f'val_r2={val_r2/n_batches:.5f}  '
            f'({time.time()-t0:.0f}s)'
        )

    return model

In [ ]:
print('Building training dataset (partitions 0–7)...')
train_ds = build_dataset_from_partitions(
    list(range(8)), FEATURE_COLS, MODEL_FEAT
)
print(f'Training sequences: {len(train_ds):,}')

print('\nBuilding validation dataset (partitions 8–9)...')
val_ds = build_dataset_from_partitions(
    [8, 9], FEATURE_COLS, MODEL_FEAT
)
print(f'Validation sequences: {len(val_ds):,}')

In [ ]:
models = []
for i in range(N_MODELS):
    print(f'\n── Training model {i+1}/{N_MODELS} (seed={SEED+i}) ──')
    torch.manual_seed(SEED + i)
    model = JaneStreetGRU(n_features=N_FEATURES).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    print(f'  Parameters: {n_params:,}')
    model = train_one_model(model, train_ds, val_ds, n_epochs=5)
    models.append(model)
    # Save checkpoint
    torch.save(model.state_dict(), f'gru_model_{i}.pt')
    print(f'  Saved → gru_model_{i}.pt')

print('\nAll models trained.')

## 6. Online Learning

This is the key differentiator of the top solutions. At the start of each new day (when `time_id == 0`), the competition API provides the **previous day's actual `responder_6` values** via the `lags` DataFrame.

We use this to do a few gradient steps on each model — fine-tuning on the most recent day's data to adapt to the current market regime. The learning rate is very small (`1e-4`) to avoid catastrophic forgetting of what was learned during training.

In [ ]:
def online_update(
    model: JaneStreetGRU,
    day_sequences: list,  # list of (X, y, w) numpy arrays from previous day
    lr: float = ONLINE_LR,
    n_steps: int = ONLINE_STEPS,
) -> None:
    """
    Fine-tune model on previous day's realized data.
    Modifies model weights in-place.

    This runs in-place because at inference time we can't afford to keep
    a copy of the original weights — we want the live model to adapt.
    """
    if not day_sequences:
        return

    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for _ in range(n_steps):
        total_loss = 0.0
        for X_np, y_np, w_np in day_sequences:
            X = torch.tensor(X_np).unsqueeze(0).to(DEVICE)  # (1, seq_len, n_feat)
            y = torch.tensor(y_np).unsqueeze(0).to(DEVICE)
            w = torch.tensor(w_np).unsqueeze(0).to(DEVICE)
            m = torch.ones_like(y, dtype=torch.bool)

            optimizer.zero_grad()
            pred, _ = model(X)
            loss = weighted_mse_loss(y, pred, w, m)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()
            total_loss += loss.item()

    model.eval()

## 7. Inference API

The competition feeds data row-by-row in real time. The inference function must:

1. **Maintain a rolling buffer** per symbol (last N_ROLL rows of raw features)
2. **Compute rolling features** from that buffer
3. **Pass through GRU with persistent hidden state** — hidden state is never reset    mid-day, only at the start of each new day
4. **At `time_id == 0`**: receive previous day's `responder_6` from the `lags` API,    run online weight updates on all models, then reset hidden states
5. **Average predictions** from all N_MODELS

The hidden state is the GRU's 'memory' of everything that has happened to this symbol today. Resetting it at day boundaries is correct because each day starts fresh with no intraday history.

In [ ]:
import kaggle_evaluation.jane_street_inference_server

# ── Global inference state ─────────────────────────────────────────────────
# Rolling raw-feature buffer: symbol_id → deque of feature vectors
from collections import deque
symbol_buffers: dict[int, deque] = defaultdict(lambda: deque(maxlen=N_ROLL))

# Per-symbol, per-model hidden states: {symbol_id: [h_model0, h_model1, ...]}
# Shape of each hidden: (num_layers, 1, hidden_size)
hidden_states: dict[int, list] = {}

# Previous-day data accumulated for online learning
# {symbol_id: {'X': [...], 'y': [...], 'w': [...]}}
prev_day_data: dict = defaultdict(lambda: {'X': [], 'y': [], 'w': []})
_prev_day_date: int = -1

[m.eval() for m in models]


def _compute_features_from_buffer(
    buffer: deque,
    current_row: np.ndarray,
) -> np.ndarray:
    """
    Append current row to buffer and compute rolling stats.
    Returns a 1D array of shape (N_FEATURES,).
    """
    buffer.append(current_row)
    arr = np.array(buffer, dtype=np.float32)     # (T, n_raw_feat)
    arr = np.nan_to_num(arr, nan=0.0)
    raw  = arr[-1]                               # most recent row
    mean = arr.mean(axis=0)
    std  = arr.std(axis=0)
    return np.concatenate([raw, mean, std])      # (3 * n_raw_feat,)


def predict(test: pl.DataFrame, lags: pl.DataFrame | None) -> pl.DataFrame:
    global _prev_day_date, hidden_states, symbol_buffers, prev_day_data

    test_pd  = test.to_pandas()
    date_id  = int(test_pd['date_id'].iloc[0])
    time_id  = int(test_pd['time_id'].iloc[0])

    # ── New day: online update + reset hidden states ──────────────────────
    if time_id == 0 and date_id != _prev_day_date:
        _prev_day_date = date_id

        # Online update: fine-tune on previous day's realized responder_6
        if lags is not None and prev_day_data:
            lags_pd = lags.to_pandas().set_index('symbol_id')
            day_seqs = []
            for sym, data in prev_day_data.items():
                if sym in lags_pd.index and len(data['X']) > 0:
                    y_realized = np.array(data['y'], dtype=np.float32)
                    # Fill in actual responder_6 from lags for last time step
                    lag_val = lags_pd.loc[sym, 'responder_6_lag_1'] \
                              if 'responder_6_lag_1' in lags_pd.columns else 0.0
                    y_realized[-1] = float(lag_val)
                    X_seq = np.stack(data['X']).astype(np.float32)
                    w_seq = np.array(data['w'], dtype=np.float32)
                    day_seqs.append((X_seq, y_realized, w_seq))

            for m in models:
                online_update(m, day_seqs)

        # Reset hidden states and previous-day buffer
        hidden_states  = {}
        prev_day_data  = defaultdict(lambda: {'X': [], 'y': [], 'w': []})

    # ── Build features for each symbol in this batch ──────────────────────
    all_preds = []

    for _, row in test_pd.iterrows():
        sym = int(row['symbol_id'])
        w   = float(row.get('weight', 1.0))

        raw_feat = row[FEATURE_COLS].fillna(0).values.astype(np.float32)
        feat_vec = _compute_features_from_buffer(symbol_buffers[sym], raw_feat)

        # Accumulate for online learning tomorrow
        prev_day_data[sym]['X'].append(feat_vec)
        prev_day_data[sym]['y'].append(0.0)  # placeholder, overwritten at day end
        prev_day_data[sym]['w'].append(w)

        # Initialize hidden state if new symbol
        if sym not in hidden_states:
            hidden_states[sym] = [
                torch.zeros(NUM_LAYERS, 1, HIDDEN_SIZE, device=DEVICE)
                for _ in models
            ]

        # Run each model and average
        sym_preds = []
        x_tensor  = torch.tensor(feat_vec).unsqueeze(0).unsqueeze(0).to(DEVICE)
                    # shape: (1, 1, N_FEATURES)

        with torch.no_grad():
            for mi, m in enumerate(models):
                pred, new_h = m(x_tensor, hidden_states[sym][mi])
                hidden_states[sym][mi] = new_h.detach()
                sym_preds.append(pred.item())

        all_preds.append(np.mean(sym_preds))

    predictions = test.select('row_id').with_columns(
        pl.Series(TARGET, np.clip(all_preds, -5.0, 5.0), dtype=pl.Float32)
    )
    return predictions


# ── Launch ─────────────────────────────────────────────────────────────────
inference_server = kaggle_evaluation.jane_street_inference_server.JSInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway((
        f'{INPUT_DIR}/test.parquet',
        f'{INPUT_DIR}/lags.parquet',
    ))